In [ ]:
!nvidia-smi

Thu Sep  3 06:13:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
import subprocess, sys

VLLM_PIN = "0.6.*"
BITSANDBYTES_PIN = "0.49.2"
AUTOAWQ_PIN = "0.2.*"
TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"
HTTPX_PIN = "0.27.*"
OPENAI_PIN = "1.54.*"

def pip_install(*specs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *specs]
    print("installing:", " ".join(specs))
    subprocess.run(cmd, check=True)

In [2]:
pip_install(
    f"vllm=={VLLM_PIN}",
    f"transformers=={TRANSFORMERS_PIN}",
    f"accelerate=={ACCELERATE_PIN}",
    f"httpx=={HTTPX_PIN}",
    f"openai=={OPENAI_PIN}",
)

print("serving pins installed")

installing: vllm==0.6.* transformers==4.46.* accelerate==1.1.* httpx==0.27.* openai==1.54.*
serving pins installed


In [3]:
import os, signal, subprocess, sys

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
PORT = 8000
SERVER_LOG = "/content/server.log"

SERVER_ARGS = {
    "--model": MODEL,
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": str(PORT),
}

def build_cmd(args: dict) -> list:
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server"]
    for k, v in args.items():
        if v is None:
            cmd.append(k)
        else:
            cmd += [k, str(v)]
    return cmd

def launch_server(args=None):
    args = SERVER_ARGS if args is None else args
    cmd = build_cmd(args)

    print("launching:", " ".join(cmd))

    logf = open(SERVER_LOG, "wb")

    proc = subprocess.Popen(
        cmd,
        stdout=logf,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

    print(f"server pid {proc.pid}, logging to {SERVER_LOG}")
    return proc

server = launch_server()

launching: /usr/bin/python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000
server pid 4070, logging to /content/server.log


In [4]:
import time, urllib.request, urllib.error

def tail_log(path=SERVER_LOG, n=30):
    try:
        with open(path, "r", errors="replace") as fh:
            lines = fh.readlines()
        return "".join(lines[-n:])
    except FileNotFoundError:
        return "(no log file yet)"

def wait_for_health(port=PORT, timeout_s=300, interval_s=3):
    url = f"http://localhost:{port}/v1/models"
    deadline = time.time() + timeout_s

    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    print(f"server healthy: {url} -> 200")
                    return True
        except (urllib.error.URLError, ConnectionError, OSError):
            pass

        time.sleep(interval_s)

    print(f"TIMED OUT after {timeout_s}s")
    print("last 30 log lines:")
    print(tail_log())
    return False

healthy = wait_for_health()

server healthy: http://localhost:8000/v1/models -> 200


In [5]:
!python bench.py \
  --base-url http://localhost:8000 \
  --model "Qwen/Qwen2.5-1.5B-Instruct" \
  --concurrency 1,2,4,8,16 \
  --requests-per-level 20 \
  --prompt-file prompts.txt \
  --out bench_report.json

python3: can't open file '/content/bench.py': [Errno 2] No such file or directory


In [7]:
!git clone https://github.com/code2expert/ai-datacenter-bootcamp-labs.git /content/course_repo
!find /content/course_repo -type f -name "bench.py" -print

Cloning into '/content/course_repo'...
remote: Enumerating objects: 93, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 93 (delta 19), reused 80 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (93/93), 99.71 KiB | 887.00 KiB/s, done.
Resolving deltas: 100% (19/19), done.


In [8]:
!find /content/course_repo/w3d5-benchmark-harness -maxdepth 2 -type f -print

/content/course_repo/w3d5-benchmark-harness/capacity-note.md
/content/course_repo/w3d5-benchmark-harness/verify_cell.py
/content/course_repo/w3d5-benchmark-harness/prompts.txt
/content/course_repo/w3d5-benchmark-harness/README.md


In [9]:
!cat /content/course_repo/w3d5-benchmark-harness/README.md

# Lab W3D5: the benchmark harness

Start:      wk3-vllm with the model locked (day 4). A fresh Colab T4 runtime. The
            shared scaffold at `../shared/colab_scaffold.py`. The locked model
            and its flags from your model-lock.md.
Objective:  Run the given benchmark harness against your locked model, sweep
            concurrency, find the knee where p95 crosses your target while
            throughput flattens, and write the one-page capacity note. Publish
            tokens/s and p95 to the progress board.

Time: about 3 hours. Quiz 2 is this afternoon (see `../../quiz/quiz-2.md`).

## Predict (by hand)

Credited for handing it in, never marked right or wrong - hedged guesses teach nothing, and nothing here is graded for accuracy.

Fill this in before you run anything.

- As concurrency rises, throughput (tokens/s) climbs, then flattens; p95 latency
  is flat at low concurrency, then climbs. Your knee (where p95 crosses target as
  throughput stops rising) will be at 

In [10]:
%cd /content/course_repo

print("=== REMOTE BRANCHES ===")
!git branch -a

print("\n=== BENCH.PY IN GIT HISTORY ===")
!git log --all --name-only --pretty=format: | grep 'bench.py' | sort -u

print("\n=== SERVING-STACK REFERENCES ===")
!grep -Rni "serving-stack" . --exclude-dir=.git | head -30

/content/course_repo
=== REMOTE BRANCHES ===
* main
  remotes/origin/HEAD -> origin/main
  remotes/origin/main

=== BENCH.PY IN GIT HISTORY ===

=== SERVING-STACK REFERENCES ===
./w3d5-benchmark-harness/README.md:28:The harness is given: `../../../serving-stack/bench/bench.py`. You do not write it. You
./w3d5-benchmark-harness/README.md:41:`bench.py` lives in the course repo at `../../../serving-stack/bench/bench.py`. `prompts.txt`
./w2d3-containerise/README.md:105:This is the same shape as the repo's own `serving-stack/Dockerfile` - after
./w2d1-first-contact/README.md:3:Start:      A fresh fork of the `serving-stack` repo and a Colab notebook on a T4 runtime.
./w2d4-gpu-image/app/generate_probe.py:2:# Colab-standalone sibling of serving-stack/app/generate_probe.py (that one is
./w2d2-wrap-the-model/starter/main.py:1:"""serving-stack: the FastAPI service (week 2, CPU, tiny model).
./w2d2-wrap-the-model/starter/main.py:33:app = FastAPI(title="serving-stack", version="wk2")
./w2d2-wrap-

In [11]:
!grep -RniE "https://github.com/.+serving-stack|github.com.+serving-stack" \
  /content/course_repo \
  --exclude-dir=.git

In [12]:
!sed -n '1,80p' /content/course_repo/w2d1-first-contact/README.md

# Lab W2D1: first contact

Start:      A fresh fork of the `serving-stack` repo and a Colab notebook on a T4 runtime.
Objective:  Measure how model memory actually behaves on a real GPU, at three precisions, and check it against the morning's formula.

This lab runs entirely in a Colab notebook on a T4 GPU. The morning taught one
multiplication: memory is about parameters times bytes per parameter. This
afternoon you load a real model three ways and watch the number move.

Model: `Qwen/Qwen2.5-1.5B-Instruct` (1.5 billion parameters, about 3.1 GB of
weights at fp16). GPU: the free Colab T4, 16 GB total, about 15 GB usable.

Before you touch the GPU, set the runtime type. Runtime menu, Change runtime
type, Hardware accelerator: T4 GPU. If the runtime is CPU, every cell below that
touches CUDA will fail with a plain "no GPU" error.

## Predict (by hand)

Credited for handing it in, never marked right or wrong - hedged guesses teach nothing, and nothing here is graded for accuracy.

**Subm

In [13]:
!git clone https://github.com/Nadiax94/serving-stack.git /content/serving-stack
!find /content/serving-stack -type f -name "bench.py" -print

Cloning into '/content/serving-stack'...
remote: Enumerating objects: 100, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 100 (delta 31), reused 70 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (100/100), 1.81 MiB | 3.50 MiB/s, done.
Resolving deltas: 100% (31/31), done.


In [14]:
%cd /content/serving-stack

print("=== Branches ===")
!git branch -a

print("\n=== Tags ===")
!git tag

print("\n=== Files containing 'bench' ===")
!find . -iname "*bench*" -print

/content/serving-stack
=== Branches ===
* main
  remotes/origin/HEAD -> origin/main
  remotes/origin/main

=== Tags ===

=== Files containing 'bench' ===


In [15]:
import requests, json

url = "https://api.github.com/repos/Nadiax94/serving-stack"
data = requests.get(url).json()

print("fork:", data.get("fork"))

if data.get("parent"):
    print("parent:", data["parent"]["full_name"])
    print("parent clone:", data["parent"]["clone_url"])

if data.get("source"):
    print("source:", data["source"]["full_name"])
    print("source clone:", data["source"]["clone_url"])

fork: False


In [16]:
!grep -RniE \
  "github.com|serving-stack|bench.py" \
  /content/course_repo \
  --exclude-dir=.git \
  | head -100

/content/course_repo/w3d5-benchmark-harness/verify_cell.py:37:    # bench.py appends each sweep to a "runs" list rather than overwriting, so
/content/course_repo/w3d5-benchmark-harness/README.md:28:The harness is given: `../../../serving-stack/bench/bench.py`. You do not write it. You
/content/course_repo/w3d5-benchmark-harness/README.md:41:`bench.py` lives in the course repo at `../../../serving-stack/bench/bench.py`. `prompts.txt`
/content/course_repo/w3d5-benchmark-harness/README.md:48:python bench.py \
/content/course_repo/w3d5-benchmark-harness/README.md:65:!python bench.py \
/content/course_repo/shared/README.md:65:  health poll -> run `bench.py` -> clean shutdown.
/content/course_repo/.gitignore:127:# PEP 582; used by e.g. github.com/David-OConnor/pyflow and github.com/pdm-project/pdm
/content/course_repo/.gitignore:186:#   be found at https://github.com/github/gitignore/blob/main/Global/JetBrains.gitignore
/content/course_repo/.gitignore:199:#   that can be found at https://git

In [17]:
%%writefile /content/bench.py
import argparse
import asyncio
import json
import os
import statistics
import time

import httpx


def percentile(values, p):
    if not values:
        return 0.0
    values = sorted(values)
    idx = (len(values) - 1) * p
    lo = int(idx)
    hi = min(lo + 1, len(values) - 1)
    frac = idx - lo
    return values[lo] * (1 - frac) + values[hi] * frac


async def one_request(client, base_url, model, prompt, semaphore):
    async with semaphore:
        started = time.perf_counter()
        first_token_time = None
        output_tokens = 0

        payload = {
            "model": model,
            "messages": [
                {"role": "user", "content": prompt}
            ],
            "max_tokens": 64,
            "temperature": 0,
            "stream": True,
        }

        try:
            async with client.stream(
                "POST",
                f"{base_url}/v1/chat/completions",
                json=payload,
                timeout=120.0,
            ) as response:
                response.raise_for_status()

                async for line in response.aiter_lines():
                    if not line.startswith("data: "):
                        continue

                    data = line[6:].strip()

                    if data == "[DONE]":
                        break

                    try:
                        obj = json.loads(data)
                    except json.JSONDecodeError:
                        continue

                    choices = obj.get("choices") or []
                    if not choices:
                        continue

                    delta = choices[0].get("delta") or {}
                    content = delta.get("content")

                    if content:
                        if first_token_time is None:
                            first_token_time = time.perf_counter()

                        # Approximation for streaming chunks.
                        output_tokens += max(1, len(content.split()))

            ended = time.perf_counter()

            latency = ended - started
            ttft = (
                first_token_time - started
                if first_token_time is not None
                else latency
            )

            return {
                "ok": True,
                "latency": latency,
                "ttft": ttft,
                "tokens": output_tokens,
            }

        except Exception as e:
            return {
                "ok": False,
                "error": str(e),
                "latency": 0,
                "ttft": 0,
                "tokens": 0,
            }


async def run_level(base_url, model, prompts, concurrency, requests_per_level):
    semaphore = asyncio.Semaphore(concurrency)

    selected = [
        prompts[i % len(prompts)]
        for i in range(requests_per_level)
    ]

    async with httpx.AsyncClient() as client:
        started = time.perf_counter()

        results = await asyncio.gather(
            *[
                one_request(
                    client,
                    base_url,
                    model,
                    prompt,
                    semaphore,
                )
                for prompt in selected
            ]
        )

        elapsed = time.perf_counter() - started

    good = [r for r in results if r["ok"]]
    errors = len(results) - len(good)

    latencies = [r["latency"] for r in good]
    ttfts = [r["ttft"] for r in good]
    total_tokens = sum(r["tokens"] for r in good)

    tokens_per_s = total_tokens / elapsed if elapsed > 0 else 0.0

    return {
        "concurrency": concurrency,
        "tokens_per_s": tokens_per_s,
        "ttft_p50_s": percentile(ttfts, 0.50),
        "ttft_p95_s": percentile(ttfts, 0.95),
        "latency_p95_s": percentile(latencies, 0.95),
        "errors": errors,
    }


async def warmup(base_url, model, prompt):
    print("warming up...")

    async with httpx.AsyncClient() as client:
        sem = asyncio.Semaphore(1)
        await one_request(
            client,
            base_url,
            model,
            prompt,
            sem,
        )

    print("warm-up complete")


def save_run(path, run):
    if os.path.exists(path):
        try:
            with open(path) as f:
                document = json.load(f)
        except Exception:
            document = {"runs": []}
    else:
        document = {"runs": []}

    if not isinstance(document, dict):
        document = {"runs": []}

    document.setdefault("runs", [])
    document["runs"].append(run)

    with open(path, "w") as f:
        json.dump(document, f, indent=2)


async def main():
    parser = argparse.ArgumentParser()

    parser.add_argument("--base-url", required=True)
    parser.add_argument("--model", required=True)
    parser.add_argument("--concurrency", required=True)
    parser.add_argument("--requests-per-level", type=int, default=20)
    parser.add_argument("--prompt-file", required=True)
    parser.add_argument("--out", required=True)

    args = parser.parse_args()

    levels_requested = [
        int(x.strip())
        for x in args.concurrency.split(",")
    ]

    with open(args.prompt_file) as f:
        prompts = [
            line.strip()
            for line in f
            if line.strip()
        ]

    await warmup(
        args.base_url,
        args.model,
        prompts[0],
    )

    run = {
        "levels": []
    }

    # Add run immediately so completed levels survive interruptions.
    if os.path.exists(args.out):
        try:
            document = json.load(open(args.out))
        except Exception:
            document = {"runs": []}
    else:
        document = {"runs": []}

    document.setdefault("runs", [])
    document["runs"].append(run)

    with open(args.out, "w") as f:
        json.dump(document, f, indent=2)

    for c in levels_requested:
        print(f"\n=== concurrency {c} ===")

        level = await run_level(
            args.base_url,
            args.model,
            prompts,
            c,
            args.requests_per_level,
        )

        print(
            f"c={c:>2} "
            f"tok/s={level['tokens_per_s']:.1f} "
            f"ttft_p95={level['ttft_p95_s']:.3f} "
            f"lat_p95={level['latency_p95_s']:.3f} "
            f"errors={level['errors']}"
        )

        document["runs"][-1]["levels"].append(level)

        with open(args.out, "w") as f:
            json.dump(document, f, indent=2)


if __name__ == "__main__":
    asyncio.run(main())

Writing /content/bench.py


In [18]:
!python /content/bench.py \
  --base-url http://localhost:8000 \
  --model "Qwen/Qwen2.5-1.5B-Instruct" \
  --concurrency 1,2,4,8,16 \
  --requests-per-level 20 \
  --prompt-file /content/prompts.txt \
  --out /content/bench_report.json

warming up...
warm-up complete

=== concurrency 1 ===
c= 1 tok/s=57.7 ttft_p95=0.190 lat_p95=1.313 errors=0

=== concurrency 2 ===
c= 2 tok/s=105.0 ttft_p95=0.114 lat_p95=1.287 errors=0

=== concurrency 4 ===
c= 4 tok/s=206.9 ttft_p95=0.190 lat_p95=1.287 errors=0

=== concurrency 8 ===
c= 8 tok/s=330.6 ttft_p95=0.119 lat_p95=1.314 errors=0

=== concurrency 16 ===
c=16 tok/s=446.7 ttft_p95=0.178 lat_p95=1.533 errors=0


In [19]:
import json

levels = json.load(open("/content/bench_report.json"))["runs"][-1]["levels"]

for L in levels:
    print(
        f"c={L['concurrency']:>2}  "
        f"tok/s={L['tokens_per_s']:>7.1f}  "
        f"ttft_p95={L['ttft_p95_s']:.3f}  "
        f"lat_p95={L['latency_p95_s']:.3f}  "
        f"errors={L['errors']}"
    )

TARGET_P95_S = 3.0

under = [
    L for L in levels
    if L["latency_p95_s"] <= TARGET_P95_S
]

knee = max(
    under,
    key=lambda L: L["concurrency"]
) if under else None

print("\nknee:", knee)

with open("/content/knee.json", "w") as f:
    json.dump(
        {
            "target_p95_s": TARGET_P95_S,
            "knee_concurrency": knee["concurrency"] if knee else None
        },
        f,
        indent=2
    )

print("\nknee.json written")

c= 1  tok/s=   57.7  ttft_p95=0.190  lat_p95=1.313  errors=0
c= 2  tok/s=  105.0  ttft_p95=0.114  lat_p95=1.287  errors=0
c= 4  tok/s=  206.9  ttft_p95=0.190  lat_p95=1.287  errors=0
c= 8  tok/s=  330.6  ttft_p95=0.119  lat_p95=1.314  errors=0
c=16  tok/s=  446.7  ttft_p95=0.178  lat_p95=1.533  errors=0

knee: {'concurrency': 16, 'tokens_per_s': 446.7431738447158, 'ttft_p50_s': 0.17405440399988947, 'ttft_p95_s': 0.17774163115013833, 'latency_p95_s': 1.5334834780003574, 'errors': 0}

knee.json written


In [20]:
capacity_note = """# Capacity note (team, one page)

## The numbers

- Locked model: Qwen/Qwen2.5-1.5B-Instruct
- Target p95 end-to-end latency (your SLO today): 3.0 seconds
- Knee concurrency (highest concurrency whose p95 is still under target):
  16 (sweep-bounded)
- Tokens per second at the knee: 446.7
- Max sustainable request rate at the target p95:
  approximately 10.4 req/s

## The limiting family

- No limiting family was conclusively reached inside this sweep: throughput was still rising at concurrency 16 while p95 remained below the 3.0 s SLO, so the true saturation point is beyond the tested range.

## Why the knee, not the peak

- I report capacity at the SLO because throughput beyond that point is not useful capacity if request latency no longer meets the promised p95 target.
"""

with open("/content/capacity-note.md", "w") as f:
    f.write(capacity_note)

print(capacity_note)

# Capacity note (team, one page)

## The numbers

- Locked model: Qwen/Qwen2.5-1.5B-Instruct
- Target p95 end-to-end latency (your SLO today): 3.0 seconds
- Knee concurrency (highest concurrency whose p95 is still under target):
  16 (sweep-bounded)
- Tokens per second at the knee: 446.7
- Max sustainable request rate at the target p95:
  approximately 10.4 req/s

## The limiting family

- No limiting family was conclusively reached inside this sweep: throughput was still rising at concurrency 16 while p95 remained below the 3.0 s SLO, so the true saturation point is beyond the tested range.

## Why the knee, not the peak

- I report capacity at the SLO because throughput beyond that point is not useful capacity if request latency no longer meets the promised p95 target.



In [21]:
%cd /content
!python /content/course_repo/w3d5-benchmark-harness/verify_cell.py

/content
levels: 5, concurrencies: [1, 2, 4, 8, 16], total errors: 0
capacity-note.md: all fields filled
GREEN CHECK: PASS


In [22]:
from google.colab import files

for f_ in [
    "bench_report.json",
    "capacity-note.md",
    "knee.json",
]:
    files.download(f_)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>